# Apply Annotation Schemas

This notebook applies the annotator-authored schemas in `annotation_results/` to `05_union_primary_filtered.tsv`, produces a new `cleaned_label` column, writes `06_cleaned_labels.tsv`, and summarizes group counts split across hate and non-hate rows.

## What this notebook does
1. Loads the filtered unioned dataset from stage 05.
2. Loads all available annotator label and glossary annotation TSVs from `annotation_results/`.
3. Aggregates annotations into annotator-level structures that can later support IAA.
4. Resolves multi-annotator disagreements by taking the union of labels.
5. Applies direct raw-target annotations, glossary-term annotations, and legacy label backfills into `cleaned_label`.
6. Exports the enriched dataset to `outputs/unioned_data/06_cleaned_labels.tsv`.
7. Reports raw group counts split by hate and non-hate, excluding `self_referential_white_supremacist`.
8. Produces a few diagnostic figures for quick inspection.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Iterable
import ast
import re

import matplotlib.pyplot as plt
import pandas as pd

WORKDIR = Path.cwd()
if not (WORKDIR / 'outputs').exists():
    WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class AnnotationConfig:
    input_path: Path = WORKDIR / 'outputs' / 'unioned_data' / '05_union_primary_filtered.tsv'
    output_path: Path = WORKDIR / 'outputs' / 'unioned_data' / '06_cleaned_labels.tsv'
    annotation_dir: Path = WORKDIR / 'annotation_results'
    glossary_reference_path: Path = WORKDIR / 'outputs' / 'glossary' / 'glossary.tsv'
    group_label_path: Path = WORKDIR / 'outputs' / 'group_labels.tsv'
    excluded_analysis_labels: tuple[str, ...] = (
        'self_referential_white_supremacist',
        'self_referential_whites_supremacist',
    )

cfg = AnnotationConfig()
cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
def normalize_token(value) -> str:
    if value is None:
        return ''
    text = str(value).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    if text in {'', 'nan', 'none', 'null', '[]'}:
        return ''
    return text


def parse_list_like(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        candidates = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except Exception:
            pass

        text = str(value).strip()
        if not text:
            return []

        if text.startswith('[') and text.endswith(']'):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    candidates = list(parsed)
                else:
                    candidates = [parsed]
            except Exception:
                candidates = [part.strip() for part in text.split('|')] if '|' in text else [text]
        elif '|' in text:
            candidates = [part.strip() for part in text.split('|')]
        elif ',' in text:
            candidates = [part.strip() for part in text.split(',')]
        else:
            candidates = [text]

    normalized = []
    for item in candidates:
        token = normalize_token(item)
        if token:
            normalized.append(token)
    return sorted(set(normalized))


def parse_annotation_label_string(value) -> list[str]:
    labels = []
    for item in parse_list_like(value):
        if item.startswith('*'):
            continue
        labels.append(item)
    return sorted(set(labels))


def infer_annotator_name(file_path: Path, suffix: str) -> str:
    name = file_path.name
    if name.endswith(suffix):
        name = name[: -len(suffix)]
    return normalize_token(name).replace(' ', '_')


def make_prefixed_label(coarse: str, fine_grained: str) -> str:
    return f"{normalize_token(coarse)}_{normalize_token(fine_grained)}"


def exact_set_agreement(long_df: pd.DataFrame, item_cols: list[str], label_col: str) -> pd.DataFrame:
    if long_df.empty or long_df['annotator'].nunique() < 2:
        return pd.DataFrame(columns=['annotator_a', 'annotator_b', 'overlap_items', 'exact_match_rate'])

    grouped = (
        long_df.groupby(item_cols + ['annotator'])[label_col]
        .apply(lambda series: tuple(sorted(set(series))))
        .reset_index()
    )

    records = []
    annotators = sorted(grouped['annotator'].unique())
    for annotator_a, annotator_b in combinations(annotators, 2):
        left = grouped[grouped['annotator'] == annotator_a].drop(columns=['annotator'])
        right = grouped[grouped['annotator'] == annotator_b].drop(columns=['annotator'])
        merged = left.merge(right, on=item_cols, how='inner', suffixes=('_a', '_b'))
        if merged.empty:
            continue
        exact_match_rate = (merged[f'{label_col}_a'] == merged[f'{label_col}_b']).mean()
        records.append({
            'annotator_a': annotator_a,
            'annotator_b': annotator_b,
            'overlap_items': len(merged),
            'exact_match_rate': round(float(exact_match_rate), 4),
        })

    if not records:
        return pd.DataFrame(columns=['annotator_a', 'annotator_b', 'overlap_items', 'exact_match_rate'])

    return pd.DataFrame(records).sort_values(['annotator_a', 'annotator_b']).reset_index(drop=True)

In [ ]:
group_labels_df = pd.read_csv(cfg.group_label_path, sep='\t')
group_labels_df['schema_label'] = group_labels_df.apply(
    lambda row: make_prefixed_label(row['coarse'], row['fine_grained']),
    axis=1,
)

legacy_target_map = {
    normalize_token(row['fine_grained']): row['schema_label']
    for _, row in group_labels_df.iterrows()
}
legacy_target_map.update({
    'woman': 'gender_women',
    'women': 'gender_women',
    'man': 'gender_men',
    'men': 'gender_men',
    'trans': 'gender_transgender_unspecified',
    'transgender': 'gender_transgender_unspecified',
    'gay': 'sexuality_gay',
    'lesbian': 'sexuality_lesbian',
    'bisexual': 'sexuality_bisexual',
    'black': 'race_black',
    'white': 'race_white',
    'asian': 'race_asian',
    'latino': 'race_latinx',
    'latinx': 'race_latinx',
    'middle eastern': 'race_middle eastern',
    'native american': 'race_native american',
    'pacific islander': 'race_pacific islander',
    'other': 'race_other',
    'jewish': 'religion_jewish',
    'muslim': 'religion_muslim',
    'christian': 'religion_christian',
    'atheist': 'religion_atheist',
    'buddhist': 'religion_buddhist',
    'hindu': 'religion_hindu',
    'mormon': 'religion_mormon',
    'immigrant': 'origin_immigrant',
    'undocumented': 'origin_undocumented',
    'specific country': 'origin_specific country',
    'children': 'age_children',
    'teenagers': 'age_teenagers',
    'young adults': 'age_young adults',
    'middle aged': 'age_middle aged',
    'seniors': 'age_seniors',
    'non binary': 'gender_non binary',
})

def map_legacy_targets_to_schema(tokens: Iterable[str]) -> list[str]:
    mapped = []
    for token in tokens:
        normalized = normalize_token(token)
        if not normalized:
            continue
        schema_label = legacy_target_map.get(normalized)
        if schema_label:
            mapped.append(schema_label)
    return sorted(set(mapped))

In [ ]:
df = pd.read_csv(cfg.input_path, sep='\t', low_memory=False)
df['binary_hate'] = pd.to_numeric(df['binary_hate'], errors='coerce').fillna(0).astype(int)
df['dataset_norm'] = df['dataset'].apply(normalize_token)
df['raw_targets_parsed'] = df['targets'].apply(parse_list_like)
df['target_std_parsed'] = df['target_std'].apply(parse_list_like)

label_files = sorted(cfg.annotation_dir.glob('*_label_annotations.tsv'))
glossary_files = sorted(cfg.annotation_dir.glob('*_glossary_annotations.tsv'))

print('Input rows:', len(df))
print('Label annotation files:', [path.name for path in label_files])
print('Glossary annotation files:', [path.name for path in glossary_files])

In [ ]:
label_annotation_rows = []
for file_path in label_files:
    annotator = infer_annotator_name(file_path, '_label_annotations.tsv')
    annotator_df = pd.read_csv(file_path, sep='\t')
    for _, row in annotator_df.iterrows():
        dataset_norm = normalize_token(row.get('dataset'))
        raw_target_label_norm = normalize_token(row.get('raw_target_label'))
        for dest_label in parse_annotation_label_string(row.get('dest_label')):
            label_annotation_rows.append({
                'annotator': annotator,
                'dataset_norm': dataset_norm,
                'raw_target_label_norm': raw_target_label_norm,
                'dest_label': dest_label,
            })

label_annotation_long = pd.DataFrame(label_annotation_rows)
if label_annotation_long.empty:
    label_annotation_long = pd.DataFrame(columns=['annotator', 'dataset_norm', 'raw_target_label_norm', 'dest_label'])

label_annotation_agg = (
    label_annotation_long.groupby(['dataset_norm', 'raw_target_label_norm'])['dest_label']
    .apply(lambda series: sorted(set(series)))
    .reset_index(name='combined_labels')
)
label_annotation_lookup = {
    (row['dataset_norm'], row['raw_target_label_norm']): row['combined_labels']
    for _, row in label_annotation_agg.iterrows()
}

print('Direct label annotation rows:', len(label_annotation_long))
print('Direct label annotation items:', len(label_annotation_agg))
display(label_annotation_agg.head(10))

label_iaa = exact_set_agreement(
    label_annotation_long,
    item_cols=['dataset_norm', 'raw_target_label_norm'],
    label_col='dest_label',
)
print('Direct label exact-set agreement across annotators:')
display(label_iaa if not label_iaa.empty else pd.DataFrame([{'note': 'IAA unavailable until at least two annotators label overlapping items.'}]))

In [ ]:
glossary_reference_df = pd.read_csv(cfg.glossary_reference_path, sep='\t')
glossary_reference_df['term_norm'] = glossary_reference_df['term'].apply(normalize_token)
glossary_reference_df['surface_form_list'] = glossary_reference_df['surface_forms'].apply(parse_list_like)
glossary_surface_lookup = {
    row['term_norm']: sorted(set(row['surface_form_list'] + [row['term_norm']]))
    for _, row in glossary_reference_df.iterrows()
    if row['term_norm']
}

glossary_annotation_rows = []
for file_path in glossary_files:
    annotator = infer_annotator_name(file_path, '_glossary_annotations.tsv')
    annotator_df = pd.read_csv(file_path, sep='\t')
    for _, row in annotator_df.iterrows():
        term_norm = normalize_token(row.get('term'))
        for inferred_target in parse_annotation_label_string(row.get('inferred_target')):
            glossary_annotation_rows.append({
                'annotator': annotator,
                'term_norm': term_norm,
                'inferred_target': inferred_target,
            })

glossary_annotation_long = pd.DataFrame(glossary_annotation_rows)
if glossary_annotation_long.empty:
    glossary_annotation_long = pd.DataFrame(columns=['annotator', 'term_norm', 'inferred_target'])

glossary_annotation_agg = (
    glossary_annotation_long.groupby('term_norm')['inferred_target']
    .apply(lambda series: sorted(set(series)))
    .reset_index(name='combined_labels')
)
glossary_annotation_agg['surface_forms'] = glossary_annotation_agg['term_norm'].apply(
    lambda term: glossary_surface_lookup.get(term, [term] if term else [])
)

print('Glossary annotation rows:', len(glossary_annotation_long))
print('Glossary annotation items:', len(glossary_annotation_agg))
display(glossary_annotation_agg.head(10))

glossary_iaa = exact_set_agreement(
    glossary_annotation_long,
    item_cols=['term_norm'],
    label_col='inferred_target',
)
print('Glossary exact-set agreement across annotators:')
display(glossary_iaa if not glossary_iaa.empty else pd.DataFrame([{'note': 'IAA unavailable until at least two annotators label overlapping glossary terms.'}]))

In [ ]:
surface_to_labels: dict[str, set[str]] = {}
surface_to_terms: dict[str, set[str]] = {}
for _, row in glossary_annotation_agg.iterrows():
    labels = set(row['combined_labels'])
    term_norm = row['term_norm']
    for surface_form in row['surface_forms']:
        normalized_surface = normalize_token(surface_form)
        if not normalized_surface:
            continue
        surface_to_labels.setdefault(normalized_surface, set()).update(labels)
        surface_to_terms.setdefault(normalized_surface, set()).add(term_norm)

compiled_glossary_pattern = None
if surface_to_labels:
    escaped_surface_forms = sorted((re.escape(item) for item in surface_to_labels), key=len, reverse=True)
    compiled_glossary_pattern = re.compile(r'(?<!\w)(?:' + '|'.join(escaped_surface_forms) + r')(?!\w)', flags=re.IGNORECASE)


def detect_glossary_labels(text: str) -> tuple[list[str], list[str]]:
    if compiled_glossary_pattern is None:
        return [], []
    labels = set()
    terms = set()
    for match in compiled_glossary_pattern.finditer(str(text).lower()):
        matched_surface = normalize_token(match.group(0))
        labels.update(surface_to_labels.get(matched_surface, set()))
        terms.update(surface_to_terms.get(matched_surface, set()))
    return sorted(labels), sorted(terms)


def lookup_direct_annotation_labels(dataset_norm: str, raw_targets: Iterable[str]) -> list[str]:
    labels = set()
    for raw_target in raw_targets:
        labels.update(label_annotation_lookup.get((dataset_norm, normalize_token(raw_target)), []))
    return sorted(labels)


def apply_cleaned_labels(row: pd.Series) -> pd.Series:
    dataset_norm = row['dataset_norm']
    raw_targets = row['raw_targets_parsed']
    legacy_labels = map_legacy_targets_to_schema(row['target_std_parsed'])
    direct_labels = lookup_direct_annotation_labels(dataset_norm, raw_targets)
    glossary_labels, matched_terms = detect_glossary_labels(row['text'])
    cleaned_labels = sorted(set(legacy_labels) | set(direct_labels) | set(glossary_labels))
    return pd.Series({
        'cleaned_label': cleaned_labels,
        'direct_annotation_label': direct_labels,
        'glossary_annotation_label': glossary_labels,
        'matched_glossary_terms': matched_terms,
        'legacy_backfill_label': legacy_labels,
    })

In [ ]:
annotation_applied = df.apply(apply_cleaned_labels, axis=1)
df = pd.concat([df, annotation_applied], axis=1)

print('Rows with any cleaned label:', int(df['cleaned_label'].apply(len).gt(0).sum()))
print('Rows with direct annotation hit:', int(df['direct_annotation_label'].apply(len).gt(0).sum()))
print('Rows with glossary annotation hit:', int(df['glossary_annotation_label'].apply(len).gt(0).sum()))
print('Rows with legacy backfill only:', int(((df['legacy_backfill_label'].apply(len) > 0) & (df['direct_annotation_label'].apply(len) == 0) & (df['glossary_annotation_label'].apply(len) == 0)).sum()))

display(df[['dataset', 'targets', 'target_std', 'cleaned_label', 'direct_annotation_label', 'glossary_annotation_label', 'matched_glossary_terms']].head(10))

In [ ]:
output_df = df.drop(columns=['dataset_norm', 'raw_targets_parsed', 'target_std_parsed'])
output_df.to_csv(cfg.output_path, sep='\t', index=False)

print(f'Wrote cleaned output to: {cfg.output_path}')
print('Output columns:')
print(output_df.columns.tolist())

In [ ]:
analysis_df = output_df[['binary_hate', 'cleaned_label']].copy()
analysis_df = analysis_df.explode('cleaned_label').dropna()
analysis_df['cleaned_label'] = analysis_df['cleaned_label'].apply(normalize_token)
analysis_df = analysis_df[analysis_df['cleaned_label'] != '']
analysis_df = analysis_df[~analysis_df['cleaned_label'].isin(set(cfg.excluded_analysis_labels))]
analysis_df['hate_bucket'] = analysis_df['binary_hate'].map({1: 'hate', 0: 'non_hate'}).fillna('non_hate')
analysis_counts = (
    analysis_df.groupby(['cleaned_label', 'hate_bucket']).size().reset_index(name='count')
)
analysis_pivot = (
    analysis_counts.pivot(index='cleaned_label', columns='hate_bucket', values='count')
    .fillna(0)
    .astype(int)
)
for column in ['hate', 'non_hate']:
    if column not in analysis_pivot.columns:
        analysis_pivot[column] = 0
analysis_pivot = analysis_pivot[['hate', 'non_hate']]
analysis_pivot['total'] = analysis_pivot.sum(axis=1)
analysis_pivot['hate_share'] = (analysis_pivot['hate'] / analysis_pivot['total']).round(4)
analysis_pivot = analysis_pivot.sort_values(['total', 'hate'], ascending=[False, False])

print('Raw counts by cleaned_label split across hate and non-hate:')
display(analysis_pivot.reset_index())

In [ ]:
plot_df = analysis_pivot.head(20).sort_values('total', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(plot_df.index, plot_df['non_hate'], label='non-hate', color='#9ecae1')
ax.barh(plot_df.index, plot_df['hate'], left=plot_df['non_hate'], label='hate', color='#de2d26')
ax.set_title('Top 20 cleaned labels by raw count')
ax.set_xlabel('Row count')
ax.set_ylabel('Cleaned label')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
hate_rank_df = analysis_pivot.sort_values(['hate', 'total'], ascending=[False, False]).head(20).sort_values('hate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(hate_rank_df.index, hate_rank_df['hate'], color='#a50f15')
ax.set_title('Top 20 cleaned labels by hate-row count')
ax.set_xlabel('Hate row count')
ax.set_ylabel('Cleaned label')
plt.tight_layout()
plt.show()

In [ ]:
coarse_counts = analysis_df.copy()
coarse_counts['coarse_group'] = coarse_counts['cleaned_label'].str.split('_', n=1).str[0]
coarse_pivot = (
    coarse_counts.groupby(['coarse_group', 'hate_bucket']).size()
    .unstack(fill_value=0)
    .reindex(columns=['non_hate', 'hate'], fill_value=0)
    .sort_values(['hate', 'non_hate'], ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
coarse_pivot.plot(kind='bar', ax=ax, color=['#9ecae1', '#de2d26'])
ax.set_title('Counts by coarse group and hate status')
ax.set_xlabel('Coarse group')
ax.set_ylabel('Row count')
ax.legend(frameon=False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(coarse_pivot.reset_index())